## Middleware

Middleware provides a way to more tightly control what happens inside an agent. Middleware is usefull for the following

 - Tracking agent's behaviour with logging, analytics and debugging
 - transforming prompts, tools selection and output formatting
 - adding retiries, fallback and early termination logic
 - applying rate limits, guardrails, and PII detection

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model("groq:qwen/qwen3.8-27b")

In [ ]:
import os
import requests
from dotenv import load_dotenv

# Load environment variables (including GROQ_API_KEY) from the .env file
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

# Authenticate via Bearer token and fetch the list of available Groq models
headers = {"Authorization": f"Bearer {api_key}"}
response = requests.get(
    "https://api.groq.com/openai/v1/models", headers=headers)

# Print each available model identifier returned by the API
for model in response.json().get("data", []):
    print(model["id"])

allam-2-7b
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
whisper-large-v3
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b


### Summarization Middleware

Automatically summarizes older conversation history when token or message thresholds are approached. It keeps recent messages intact and pairs AI and tool messages together so long agent loops don't overflow the context window.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message based summarization
agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b", trigger=("messages", 10), keep=("messages", 4)
        )
    ],
)

In [ ]:
# Run with thread id
config = {"configurable": {"thread_id": "test-1"}}

In [ ]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='60fbdd92-365c-4344-9f07-2137601e8d59'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 19, 'total_tokens': 27, 'completion_time': 0.032461266, 'completion_tokens_details': None, 'prompt_time': 2.107e-05, 'prompt_tokens_details': None, 'queue_time': 0.057770039, 'total_time': 0.032482336}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_2510d2d224', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a21682-0896-7592-9e70-4f794ee8e5c2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 8, 'total_tokens': 27})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='60fbdd92-365c-4344-9f07-2137601e8d59'), AIMessage(conten

### based on token size

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token Counter(Approx)


def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [9]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~137 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='a57d783d-2851-42e9-8527-a14a348d29c3'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ft0rqep31', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 279, 'total_tokens': 306, 'completion_time': 0.152864084, 'completion_tokens_details': None, 'prompt_time': 0.030368848, 'prompt_tokens_details': None, 'queue_time': 0.063130311, 'total_time': 0.183232932}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_3f7cb2e2dd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a21696-69b7-7656-a6c6-55e74abe1910-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'ft0rqep31', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'in

### based on fraction

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"


# LOW fraction for testing!
# Both Qwen 3 (Groq) and gpt-oss-20b (Groq) use a 131,072 token context window
agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            # 0.488% of 131,072 = ~640 tokens (0.005 = 655 tokens)
            trigger=("fraction", 0.00488),
            # 0.195% of 131,072 = ~256 tokens (0.002 = 262 tokens)
            keep=("fraction", 0.00195),
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}


# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4


# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 131072  # 131,072 context window for Groq Qwen 3 / gpt-oss-20b
    print(
        f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~74 tokens (0.0565%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='c545ab55-167d-45ec-ab45-6e4c9285837e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '61e64sqbt', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 271, 'total_tokens': 298, 'completion_time': 0.070150975, 'completion_tokens_details': None, 'prompt_time': 0.02114176, 'prompt_tokens_details': None, 'queue_time': 0.054373477, 'total_time': 0.091292735}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a2169d-1fb6-7323-a1f5-9c2b2fd0d30d-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '61e64sqbt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp

### Human In The Loop Middleware

The Human-in-the-Loop (HITL) middleware lets you add human oversight to agent tool calls. When a model proposes an action that might require review—for example, writing to a file or executing SQL—the middleware can pause execution and wait for a decision.
It does this by checking each tool call against a configurable policy. If intervention is needed, the middleware issues an interrupt that halts execution. The graph state is saved using LangGraph’s persistence layer, so execution can pause safely and resume later.
A human decision then determines what happens next: the action can be approved as-is (approve), modified before running (edit), rejected with feedback (reject), or responded to directly (respond) for “ask user” style tools.

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [ ]:
agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
            }
        )
    ]
)

In [25]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(
        content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [26]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='4b505cb1-d001-4c19-9c66-eb2dbe1cd42a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'n60j9ywjh', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.172466978, 'completion_tokens_details': None, 'prompt_time': 0.028496201, 'prompt_tokens_details': None, 'queue_time': 0.064057029, 'total_time': 0.200963179}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a216a9-8af9-769c-a9c4-f3196475dd06-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': '

In [ ]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email send was not executed, as the tool call was rejected. Let me know if you'd like me to try again or if there's anything I can help you with instead.


### Impelementing a chatbot for sending moc email and accepting, rejecting or editing them using basic python input